In [1]:
from pathlib import Path

import pandas as pd

In [2]:
ROOT = Path.cwd()
DATA_PATH = ROOT / "metrics/ohare_filtered/1768186260"
AGG_E2E_PATH = DATA_PATH / "aggregated_e2e_metrics.csv"
AGG_MATCH_PATH = DATA_PATH / "aggregated_match_metrics.csv"
AGG_PROM_PATH = DATA_PATH / "aggregated_prometheus_metrics.csv"


In [3]:
def join_metrics(
    *,
    e2e_steps_df: pd.DataFrame,
    prom_df: pd.DataFrame,
    match_df: pd.DataFrame,
    run_key: str = "run_id",
) -> pd.DataFrame:
    """Join E2E, Prometheus, and matching metrics into unified table."""

    # Normalize run_id across dataframes
    def normalize_ids(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # pega uma coluna fonte existente
        if run_key in df.columns:
            src = run_key
        elif "experiment_id" in df.columns:
            src = "experiment_id"
        elif "run_id" in df.columns:
            src = "run_id"
        else:
            raise ValueError(f"Missing '{run_key}', 'experiment_id' or 'run_id'")

        # garante que df[src] é 1D (evita o ValueError: 2)
        if df.columns.duplicated().any():
            df = df.loc[:, ~df.columns.duplicated()]

        df["experiment_id"] = df[src]
        df["run_id"] = df[src]

        return df

    e2e = normalize_ids(e2e_steps_df).copy()
    prom = normalize_ids(prom_df).copy()
    mm = normalize_ids(match_df).copy()

    # Validate E2E columns
    required = {
        run_key,
        "matcher_name",
        "mode",
        "step_index",
        "timestamp",
        "step_latency_ms",
        "inter_arrival_ms",
        "points_per_step",
        "cum_points",
        "memory_mb",
        "cpu_time_ms",
    }
    missing = required - set(e2e.columns)
    if missing:
        raise ValueError(f"E2E dataframe missing columns: {sorted(missing)}")

    e2e = e2e.sort_values([run_key, "timestamp", "step_index"])

    # Aggregate E2E to run-level
    e2e_summary = _aggregate_e2e_metrics(e2e, run_key)

    # Join all metrics
    runs = prom.merge(mm, on=run_key, how="left", suffixes=("_prom", "_mm"))
    runs["run_id"] = runs["run_id_prom"]

    runs = runs.merge(e2e_summary, on=run_key, how="left", suffixes=("", "_e2e"))

    cols = [
        "duration_s",
        "cpu_avg",
        "cpu_max",
        "cpu_total_core_s",
        "mem_avg_mb",
        "mem_peak_mb",
        "rx_avg_bps",
        "tx_avg_bps",
        "rx_peak_bps",
        "tx_peak_bps",
        "rx_total_bytes",
        "tx_total_bytes",
        "matcher_name_prom",
        "mode_prom",
        "dataset_id_prom",
        "run_id",
        "vehicle_id_prom",
        "adapter_prom",
        "batch_size_prom",
        "gps_accuracy_prom",
        "precision",
        "recall",
        "f1_score",
        "error_rate",
        "accuracy",
        "newson_krumm_error",
        "matched_count",
        "added_count",
        "missing_count",
        "ground_truth_count",
        "total_gt_length",
        "total_added_length",
        "total_missing_length",
        "vehicle_id_mm",
        "experiment_id",
        "dataset_id_mm",
        "matcher_name_mm",
        "mode_mm",
        "adapter_mm",
        "batch_size_mm",
        "gps_accuracy_mm",
        "matcher_name",
        "mode",
        "e2e_t0",
        "e2e_t1",
        "e2e_steps",
        "e2e_total_points",
        "e2e_execution_time_s",
        "e2e_throughput_pps",
        "e2e_avg_step_latency_ms",
        "e2e_p50_step_latency_ms",
        "e2e_p75_step_latency_ms",
        "e2e_p95_step_latency_ms",
        "e2e_p99_step_latency_ms",
        "e2e_max_step_latency_ms",
        "e2e_avg_inter_arrival_ms",
        "e2e_p50_inter_arrival_ms",
        "e2e_p75_inter_arrival_ms",
        "e2e_p95_inter_arrival_ms",
        "e2e_p99_inter_arrival_ms",
        "e2e_max_inter_arrival_ms",
        "e2e_avg_points_per_step",
        "e2e_client_mem_peak_mb",
        "e2e_client_cpu_time_ms",
    ]
    # ensure column order
    runs = runs[cols]

    return runs


@staticmethod
def _aggregate_e2e_metrics(e2e: pd.DataFrame, run_key: str) -> pd.DataFrame:
    """Aggregate step-level E2E metrics to run-level."""

    # Temporal bounds and throughput
    e2e_bounds = e2e.groupby(run_key, as_index=False).agg(
        e2e_t0=("timestamp", "min"),
        e2e_t1=("timestamp", "max"),
        e2e_steps=("step_index", "count"),
        e2e_total_points=("cum_points", "max"),
    )
    e2e_bounds["e2e_execution_time_s"] = e2e_bounds["e2e_t1"] - e2e_bounds["e2e_t0"]
    e2e_bounds["e2e_throughput_pps"] = e2e_bounds["e2e_total_points"] / e2e_bounds[
        "e2e_execution_time_s"
    ].replace(0, pd.NA)

    # Distribution statistics
    e2e_stats = e2e.groupby(run_key, as_index=False).agg(
        e2e_avg_step_latency_ms=("step_latency_ms", "mean"),
        e2e_p50_step_latency_ms=("step_latency_ms", lambda s: s.quantile(0.50)),
        e2e_p75_step_latency_ms=("step_latency_ms", lambda s: s.quantile(0.75)),
        e2e_p95_step_latency_ms=("step_latency_ms", lambda s: s.quantile(0.95)),
        e2e_p99_step_latency_ms=("step_latency_ms", lambda s: s.quantile(0.99)),
        e2e_max_step_latency_ms=("step_latency_ms", "max"),
        e2e_avg_inter_arrival_ms=("inter_arrival_ms", "mean"),
        e2e_p50_inter_arrival_ms=("inter_arrival_ms", lambda s: s.quantile(0.50)),
        e2e_p75_inter_arrival_ms=("inter_arrival_ms", lambda s: s.quantile(0.75)),
        e2e_p95_inter_arrival_ms=("inter_arrival_ms", lambda s: s.quantile(0.95)),
        e2e_p99_inter_arrival_ms=("inter_arrival_ms", lambda s: s.quantile(0.99)),
        e2e_max_inter_arrival_ms=("inter_arrival_ms", "max"),
        e2e_avg_points_per_step=("points_per_step", "mean"),
        e2e_client_mem_peak_mb=("memory_mb", "max"),
        e2e_client_cpu_time_ms=("cpu_time_ms", "sum"),
    )

    # Run identifiers
    e2e_id = e2e.groupby(run_key, as_index=False).agg(
        matcher_name=("matcher_name", "first"),
        mode=("mode", "first"),
    )

    return e2e_id.merge(e2e_bounds, on=run_key, how="inner").merge(
        e2e_stats, on=run_key, how="inner"
    )

In [4]:
agg_e2e_df = pd.read_csv(AGG_E2E_PATH)
agg_e2e_df

,run_id,matcher_name,mode,step_index,timestamp,step_latency_ms,inter_arrival_ms,points_per_step,cum_points,memory_mb,cpu_time_ms
0,16523baf49f64af2920f43fbc652ea15,graphium,online,0,1.768186e+09,1192.833003,NaN,30,30,314.90625,10.0
1,16523baf49f64af2920f43fbc652ea15,graphium,online,1,1.768186e+09,182.609144,182.775974,5,35,314.90625,0.0
2,16523baf49f64af2920f43fbc652ea15,graphium,online,2,1.768186e+09,178.779128,178.938150,5,40,314.90625,0.0
3,16523baf49f64af2920f43fbc652ea15,graphium,online,3,1.768186e+09,179.915613,180.033445,5,45,314.90625,0.0
4,16523baf49f64af2920f43fbc652ea15,graphium,online,4,1.768186e+09,218.816028,218.941450,5,50,314.90625,0.0
...,...,...,...,...,...,...,...,...,...,...,...
14184,15a74036e9fe4f108949b14626cde04e,osrm,offline,2,1.768199e+09,3344.208166,3344.361305,5,15,338.75000,10.0
14185,15a74036e9fe4f108949b14626cde04e,osrm,offline,3,1.768199e+09,0.185594,0.252724,0,15,338.75000,0.0
14186,672439444d6944ccace434e680f85e34,osrm,offline,0,1.768199e+09,6018.144771,NaN,10,10,338.90625,0.0
14187,672439444d6944ccace434e680f85e34,osrm,offline,1,1.768199e+09,3344.282321,3344.424009,5,15,338.90625,10.0


In [5]:
agg_match_df = pd.read_csv(AGG_MATCH_PATH)
agg_match_df

,run_id,precision,recall,f1_score,error_rate,accuracy,newson_krumm_error,matched_count,added_count,missing_count,...,total_added_length,total_missing_length,vehicle_id,experiment_id,dataset_id,matcher_name,mode,adapter,batch_size,gps_accuracy
0,9452b53d65ad46e7bca784e891330c42,0.972407,0.913960,0.942278,0.057722,0.890856,0.111975,10,1,2,...,17.812997,59.095339,1139,16523baf49f64af2920f43fbc652ea15,ohare_filtered_vid_1139_sr_1.0,graphium,online,native,5.0,NaN
1,318c151658d24539b965db7f45735ab1,0.972407,0.913960,0.942278,0.057722,0.890856,0.111975,10,1,2,...,17.812997,59.095339,1139,324ea397c7a54ad18e1339fa50aed98c,ohare_filtered_vid_1139_sr_1.0,graphium,online,native,10.0,NaN
2,071f4bfd3a6e4ec488709d8baa97955b,0.972407,0.913960,0.942278,0.057722,0.890856,0.111975,10,1,2,...,17.812997,59.095339,1139,565006ebd2e74bf4945c81eb16ab6a1b,ohare_filtered_vid_1139_sr_1.0,graphium,online,native,30.0,NaN
3,38862d8619234220aa348dcdeb4000e0,0.956736,1.000000,0.977890,0.022110,0.956736,0.045220,12,2,0,...,31.059085,0.000000,1139,44a75e620a77492daa67aaf769ae6d9c,ohare_filtered_vid_1139_sr_1.0,barefoot,online,native,NaN,NaN
4,0ce2d2b28c90408f8c0bba7c7d41fac8,0.609645,0.901038,0.727238,0.272762,0.571386,0.675896,10,7,2,...,396.259501,67.970605,1139,a06171e307184813a754e7f5924c407e,ohare_filtered_vid_1139_sr_1.0,graphhopper,offline,batches,5.0,50.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
785,b287c480aac74ff880eb803fe05ca937,0.961322,0.869725,0.913232,0.086768,0.840320,0.165268,21,2,8,...,85.867008,319.680124,1423,18ce1cb4394346febcc7ae922852a78c,ohare_filtered_vid_1423_sr_20.0,graphhopper,offline,batches,10.0,50.0
786,1231a2a5a61c4380804ba1984ec75373,0.963249,0.917145,0.939631,0.060369,0.886137,0.117848,24,2,5,...,85.867008,203.317191,1423,e9ed28d1b51844e09b87b7243679b2f7,ohare_filtered_vid_1423_sr_20.0,graphhopper,offline,batches,30.0,50.0
787,90a68f9b77c4457daf6e1f81e35ded83,0.976326,0.801149,0.880105,0.119895,0.785882,0.218277,18,1,11,...,47.669382,487.956680,1423,15a74036e9fe4f108949b14626cde04e,ohare_filtered_vid_1423_sr_20.0,osrm,offline,batches,5.0,NaN
788,5f44a93a1ec54afe903b447df21c8e38,0.976326,0.801149,0.880105,0.119895,0.785882,0.218277,18,1,11,...,47.669382,487.956680,1423,672439444d6944ccace434e680f85e34,ohare_filtered_vid_1423_sr_20.0,osrm,offline,batches,10.0,NaN


In [6]:
agg_prom_df = pd.read_csv(AGG_PROM_PATH)
agg_prom_df

,duration_s,cpu_avg,cpu_max,cpu_total_core_s,mem_avg_mb,mem_peak_mb,rx_avg_bps,tx_avg_bps,rx_peak_bps,tx_peak_bps,rx_total_bytes,tx_total_bytes,matcher_name,mode,dataset_id,experiment_id,vehicle_id,adapter,batch_size,gps_accuracy
0,15.0,1.643418,2.476856,24.900649,1433.749512,1492.902344,1225.355153,1726.482962,2708.800000,4123.350000,19605.682452,2.762373e+04,graphium,online,ohare_filtered_vid_1139_sr_1.0,16523baf49f64af2920f43fbc652ea15,1139,native,5.0,NaN
1,14.0,1.745557,2.935444,24.904509,1439.782813,1491.480469,709.491711,657.340019,1297.477943,1389.500000,10642.375661,9.860100e+03,graphium,online,ohare_filtered_vid_1139_sr_1.0,324ea397c7a54ad18e1339fa50aed98c,1139,native,10.0,NaN
2,14.0,1.653615,2.772860,23.480448,1446.583854,1493.328125,563.937348,452.448820,1118.050000,1002.225000,8459.060219,6.786732e+03,graphium,online,ohare_filtered_vid_1139_sr_1.0,565006ebd2e74bf4945c81eb16ab6a1b,1139,native,30.0,NaN
3,7.0,0.378917,0.426825,2.665077,195.839844,221.234375,16946.145207,152380.020378,19332.196681,209615.798802,135569.161659,1.219040e+06,barefoot,online,ohare_filtered_vid_1139_sr_1.0,44a75e620a77492daa67aaf769ae6d9c,1139,native,NaN,NaN
4,12.0,0.385815,0.726321,4.518585,295.956771,349.316406,1805.716314,987.939743,3221.175000,1865.600000,23474.312085,1.284322e+04,graphhopper,offline,ohare_filtered_vid_1139_sr_1.0,a06171e307184813a754e7f5924c407e,1139,batches,5.0,50.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
785,18.0,0.274625,0.615558,4.643612,334.392578,357.375000,287.340894,94.898796,414.849091,228.433333,5459.476982,1.803077e+03,graphhopper,offline,ohare_filtered_vid_1423_sr_20.0,18ce1cb4394346febcc7ae922852a78c,1423,batches,10.0,50.0
786,18.0,0.239055,0.732729,4.100579,296.030469,313.644531,138.884052,5.228701,299.675724,14.303416,2638.796985,9.934533e+01,graphhopper,offline,ohare_filtered_vid_1423_sr_20.0,e9ed28d1b51844e09b87b7243679b2f7,1423,batches,30.0,50.0
787,13.0,0.000343,0.000573,0.004408,3.881836,5.074219,341.472671,573.074571,560.527258,976.078746,4780.617387,8.023044e+03,osrm,offline,ohare_filtered_vid_1423_sr_20.0,15a74036e9fe4f108949b14626cde04e,1423,batches,5.0,NaN
788,12.0,0.000632,0.001066,0.007152,3.032031,3.917969,296.743839,518.034971,459.656377,884.038180,3857.669907,6.734455e+03,osrm,offline,ohare_filtered_vid_1423_sr_20.0,672439444d6944ccace434e680f85e34,1423,batches,10.0,NaN


In [7]:
e2e_summary = _aggregate_e2e_metrics(agg_e2e_df, run_key="run_id")
e2e_summary

,run_id,matcher_name,mode,e2e_t0,e2e_t1,e2e_steps,e2e_total_points,e2e_execution_time_s,e2e_throughput_pps,e2e_avg_step_latency_ms,...,e2e_max_step_latency_ms,e2e_avg_inter_arrival_ms,e2e_p50_inter_arrival_ms,e2e_p75_inter_arrival_ms,e2e_p95_inter_arrival_ms,e2e_p99_inter_arrival_ms,e2e_max_inter_arrival_ms,e2e_avg_points_per_step,e2e_client_mem_peak_mb,e2e_client_cpu_time_ms
0,0020fcef10544182a0e5f7c4cb43c341,graphium,online,1.768187e+09,1.768187e+09,1,9,0.000000,<NA>,2889.085354,...,2889.085354,NaN,NaN,NaN,NaN,NaN,NaN,9.000000,327.242188,0.0
1,008a809f73224ea196f0809f3cee1c16,barefoot,online,1.768188e+09,1.768188e+09,10,10,6.001042,1.666377,602.874956,...,672.090135,666.782485,667.301178,669.757366,671.806431,672.159595,672.247887,1.000000,330.324219,20.0
2,00c1da53894447efbabdac5578564012,graphhopper,offline,1.768198e+09,1.768198e+09,2,16,4.025310,3.974849,5082.755121,...,6140.337516,4025.310278,4025.310278,4025.310278,4025.310278,4025.310278,4025.310278,8.000000,338.800781,10.0
3,0110eca2919843be97e7156042d8fe46,barefoot,online,1.768189e+09,1.768189e+09,52,52,8.567882,6.069178,165.019902,...,172.908286,167.997687,168.479204,169.783354,172.274470,172.737241,173.085690,1.000000,331.113281,80.0
4,01e4f29d636e41478c336ed44451c273,graphium,online,1.768189e+09,1.768189e+09,78,362,11.894533,30.43415,167.648012,...,1192.498109,154.474453,176.978111,177.850246,181.010580,191.907635,199.342012,4.641026,331.730469,180.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
785,fd86d8310bab4d62a37e85a5178a032f,graphhopper,offline,1.768199e+09,1.768199e+09,4,15,6.732541,2.227985,2376.159071,...,3376.064231,2244.180441,3355.821133,3366.029024,3374.195337,3375.828600,3376.236916,3.750000,338.750000,10.0
786,fdb4ad83523b497cabc9b48ec7d30988,graphium,online,1.768191e+09,1.768191e+09,9,257,8.094430,31.750227,1023.986403,...,1136.040574,1011.803776,1050.868273,1090.777993,1121.894360,1133.359241,1136.225462,28.555556,332.339844,70.0
787,fe0fe33ce13447dfbc49a1a1c643009c,barefoot,online,1.768188e+09,1.768188e+09,127,127,4.354653,29.164204,34.330934,...,41.789400,34.560741,34.628153,36.103606,38.777769,40.921926,41.979790,1.000000,331.937500,240.0
788,fe4cd719aaf541c7a4d787af0411fc9c,osrm,offline,1.768187e+09,1.768187e+09,2,12,0.670282,17.902922,1843.706685,...,3017.321323,670.281649,670.281649,670.281649,670.281649,670.281649,670.281649,6.000000,322.968750,10.0


In [8]:
len(set(agg_prom_df.experiment_id) & set(agg_match_df.run_id))

0

In [9]:
agg_match_df[["dataset_id", "batch_size", "matcher_name", "mode"]].drop_duplicates()

,dataset_id,batch_size,matcher_name,mode
0,ohare_filtered_vid_1139_sr_1.0,5.0,graphium,online
1,ohare_filtered_vid_1139_sr_1.0,10.0,graphium,online
2,ohare_filtered_vid_1139_sr_1.0,30.0,graphium,online
3,ohare_filtered_vid_1139_sr_1.0,NaN,barefoot,online
4,ohare_filtered_vid_1139_sr_1.0,5.0,graphhopper,offline
...,...,...,...,...
785,ohare_filtered_vid_1423_sr_20.0,10.0,graphhopper,offline
786,ohare_filtered_vid_1423_sr_20.0,30.0,graphhopper,offline
787,ohare_filtered_vid_1423_sr_20.0,5.0,osrm,offline
788,ohare_filtered_vid_1423_sr_20.0,10.0,osrm,offline


In [10]:
agg_prom_df[["dataset_id", "batch_size", "matcher_name", "mode"]].drop_duplicates()

,dataset_id,batch_size,matcher_name,mode
0,ohare_filtered_vid_1139_sr_1.0,5.0,graphium,online
1,ohare_filtered_vid_1139_sr_1.0,10.0,graphium,online
2,ohare_filtered_vid_1139_sr_1.0,30.0,graphium,online
3,ohare_filtered_vid_1139_sr_1.0,NaN,barefoot,online
4,ohare_filtered_vid_1139_sr_1.0,5.0,graphhopper,offline
...,...,...,...,...
785,ohare_filtered_vid_1423_sr_20.0,10.0,graphhopper,offline
786,ohare_filtered_vid_1423_sr_20.0,30.0,graphhopper,offline
787,ohare_filtered_vid_1423_sr_20.0,5.0,osrm,offline
788,ohare_filtered_vid_1423_sr_20.0,10.0,osrm,offline


In [11]:
agg_match_df = agg_match_df.merge(
    agg_prom_df,
    on=["dataset_id", "batch_size", "matcher_name", "mode"],
    how="inner",
    suffixes=("", "_prom"),
)[
    agg_match_df.columns.tolist()
    
]
agg_match_df

,run_id,precision,recall,f1_score,error_rate,accuracy,newson_krumm_error,matched_count,added_count,missing_count,...,total_added_length,total_missing_length,vehicle_id,experiment_id,dataset_id,matcher_name,mode,adapter,batch_size,gps_accuracy
0,9452b53d65ad46e7bca784e891330c42,0.972407,0.913960,0.942278,0.057722,0.890856,0.111975,10,1,2,...,17.812997,59.095339,1139,16523baf49f64af2920f43fbc652ea15,ohare_filtered_vid_1139_sr_1.0,graphium,online,native,5.0,NaN
1,318c151658d24539b965db7f45735ab1,0.972407,0.913960,0.942278,0.057722,0.890856,0.111975,10,1,2,...,17.812997,59.095339,1139,324ea397c7a54ad18e1339fa50aed98c,ohare_filtered_vid_1139_sr_1.0,graphium,online,native,10.0,NaN
2,071f4bfd3a6e4ec488709d8baa97955b,0.972407,0.913960,0.942278,0.057722,0.890856,0.111975,10,1,2,...,17.812997,59.095339,1139,565006ebd2e74bf4945c81eb16ab6a1b,ohare_filtered_vid_1139_sr_1.0,graphium,online,native,30.0,NaN
3,38862d8619234220aa348dcdeb4000e0,0.956736,1.000000,0.977890,0.022110,0.956736,0.045220,12,2,0,...,31.059085,0.000000,1139,44a75e620a77492daa67aaf769ae6d9c,ohare_filtered_vid_1139_sr_1.0,barefoot,online,native,NaN,NaN
4,0ce2d2b28c90408f8c0bba7c7d41fac8,0.609645,0.901038,0.727238,0.272762,0.571386,0.675896,10,7,2,...,396.259501,67.970605,1139,a06171e307184813a754e7f5924c407e,ohare_filtered_vid_1139_sr_1.0,graphhopper,offline,batches,5.0,50.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
785,b287c480aac74ff880eb803fe05ca937,0.961322,0.869725,0.913232,0.086768,0.840320,0.165268,21,2,8,...,85.867008,319.680124,1423,18ce1cb4394346febcc7ae922852a78c,ohare_filtered_vid_1423_sr_20.0,graphhopper,offline,batches,10.0,50.0
786,1231a2a5a61c4380804ba1984ec75373,0.963249,0.917145,0.939631,0.060369,0.886137,0.117848,24,2,5,...,85.867008,203.317191,1423,e9ed28d1b51844e09b87b7243679b2f7,ohare_filtered_vid_1423_sr_20.0,graphhopper,offline,batches,30.0,50.0
787,90a68f9b77c4457daf6e1f81e35ded83,0.976326,0.801149,0.880105,0.119895,0.785882,0.218277,18,1,11,...,47.669382,487.956680,1423,15a74036e9fe4f108949b14626cde04e,ohare_filtered_vid_1423_sr_20.0,osrm,offline,batches,5.0,NaN
788,5f44a93a1ec54afe903b447df21c8e38,0.976326,0.801149,0.880105,0.119895,0.785882,0.218277,18,1,11,...,47.669382,487.956680,1423,672439444d6944ccace434e680f85e34,ohare_filtered_vid_1423_sr_20.0,osrm,offline,batches,10.0,NaN


In [12]:
len(set(agg_prom_df.experiment_id) & set(agg_match_df.experiment_id))

790

In [13]:
len(set(agg_prom_df.experiment_id) & set(agg_e2e_df.run_id))

790

In [14]:
agg_prom_df.merge(agg_match_df, left_on="experiment_id", right_on="experiment_id", how="inner", suffixes=("_prom", "_mm"))

,duration_s,cpu_avg,cpu_max,cpu_total_core_s,mem_avg_mb,mem_peak_mb,rx_avg_bps,tx_avg_bps,rx_peak_bps,tx_peak_bps,...,total_gt_length,total_added_length,total_missing_length,vehicle_id_mm,dataset_id_mm,matcher_name_mm,mode_mm,adapter_mm,batch_size_mm,gps_accuracy_mm
0,15.0,1.643418,2.476856,24.900649,1433.749512,1492.902344,1225.355153,1726.482962,2708.800000,4123.350000,...,686.836705,17.812997,59.095339,1139,ohare_filtered_vid_1139_sr_1.0,graphium,online,native,5.0,NaN
1,14.0,1.745557,2.935444,24.904509,1439.782813,1491.480469,709.491711,657.340019,1297.477943,1389.500000,...,686.836705,17.812997,59.095339,1139,ohare_filtered_vid_1139_sr_1.0,graphium,online,native,10.0,NaN
2,14.0,1.653615,2.772860,23.480448,1446.583854,1493.328125,563.937348,452.448820,1118.050000,1002.225000,...,686.836705,17.812997,59.095339,1139,ohare_filtered_vid_1139_sr_1.0,graphium,online,native,30.0,NaN
3,7.0,0.378917,0.426825,2.665077,195.839844,221.234375,16946.145207,152380.020378,19332.196681,209615.798802,...,686.836705,31.059085,0.000000,1139,ohare_filtered_vid_1139_sr_1.0,barefoot,online,native,NaN,NaN
4,12.0,0.385815,0.726321,4.518585,295.956771,349.316406,1805.716314,987.939743,3221.175000,1865.600000,...,686.836705,396.259501,67.970605,1139,ohare_filtered_vid_1139_sr_1.0,graphhopper,offline,batches,5.0,50.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
785,18.0,0.274625,0.615558,4.643612,334.392578,357.375000,287.340894,94.898796,414.849091,228.433333,...,2453.878752,85.867008,319.680124,1423,ohare_filtered_vid_1423_sr_20.0,graphhopper,offline,batches,10.0,50.0
786,18.0,0.239055,0.732729,4.100579,296.030469,313.644531,138.884052,5.228701,299.675724,14.303416,...,2453.878752,85.867008,203.317191,1423,ohare_filtered_vid_1423_sr_20.0,graphhopper,offline,batches,30.0,50.0
787,13.0,0.000343,0.000573,0.004408,3.881836,5.074219,341.472671,573.074571,560.527258,976.078746,...,2453.878752,47.669382,487.956680,1423,ohare_filtered_vid_1423_sr_20.0,osrm,offline,batches,5.0,NaN
788,12.0,0.000632,0.001066,0.007152,3.032031,3.917969,296.743839,518.034971,459.656377,884.038180,...,2453.878752,47.669382,487.956680,1423,ohare_filtered_vid_1423_sr_20.0,osrm,offline,batches,10.0,NaN


In [17]:
joined_df = join_metrics(e2e_steps_df=agg_e2e_df, prom_df=agg_prom_df, match_df=agg_match_df, run_key="experiment_id")
joined_df

,duration_s,cpu_avg,cpu_max,cpu_total_core_s,mem_avg_mb,mem_peak_mb,rx_avg_bps,tx_avg_bps,rx_peak_bps,tx_peak_bps,...,e2e_max_step_latency_ms,e2e_avg_inter_arrival_ms,e2e_p50_inter_arrival_ms,e2e_p75_inter_arrival_ms,e2e_p95_inter_arrival_ms,e2e_p99_inter_arrival_ms,e2e_max_inter_arrival_ms,e2e_avg_points_per_step,e2e_client_mem_peak_mb,e2e_client_cpu_time_ms
0,15.0,1.643418,2.476856,24.900649,1433.749512,1492.902344,1225.355153,1726.482962,2708.800000,4123.350000,...,1192.833003,175.478318,178.727388,180.033445,190.009069,213.154974,218.941450,6.222222,314.906250,50.0
1,14.0,1.745557,2.935444,24.904509,1439.782813,1491.480469,709.491711,657.340019,1297.477943,1389.500000,...,1190.052957,326.990181,351.133585,352.983952,386.070156,402.083569,406.086922,11.200000,328.164062,40.0
2,14.0,1.653615,2.772860,23.480448,1446.583854,1493.328125,563.937348,452.448820,1118.050000,1002.225000,...,1185.863303,955.418507,1043.594599,1050.235868,1055.548882,1056.611485,1056.877136,28.000000,328.164062,30.0
3,7.0,0.378917,0.426825,2.665077,195.839844,221.234375,16946.145207,152380.020378,19332.196681,209615.798802,...,39.427868,34.580809,34.567356,35.793304,37.380576,38.146210,39.592743,1.000000,328.476562,200.0
4,12.0,0.385815,0.726321,4.518585,295.956771,349.316406,1805.716314,987.939743,3221.175000,1865.600000,...,238.575762,177.236568,181.450248,182.985961,188.821447,193.970988,195.305347,4.869565,328.476562,90.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
785,18.0,0.274625,0.615558,4.643612,334.392578,357.375000,287.340894,94.898796,414.849091,228.433333,...,6122.761888,3362.779856,3362.779856,3362.779856,3362.779856,3362.779856,3362.779856,7.500000,338.750000,10.0
786,18.0,0.239055,0.732729,4.100579,296.030469,313.644531,138.884052,5.228701,299.675724,14.303416,...,9483.204848,NaN,NaN,NaN,NaN,NaN,NaN,15.000000,338.750000,10.0
787,13.0,0.000343,0.000573,0.004408,3.881836,5.074219,341.472671,573.074571,560.527258,976.078746,...,3344.208166,2229.577700,3344.119072,3344.240189,3344.337082,3344.356461,3344.361305,3.750000,338.750000,10.0
788,12.0,0.000632,0.001066,0.007152,3.032031,3.917969,296.743839,518.034971,459.656377,884.038180,...,6018.144771,3344.424009,3344.424009,3344.424009,3344.424009,3344.424009,3344.424009,7.500000,338.906250,10.0


In [20]:
len(joined_df[joined_df.experiment_id == joined_df.run_id]), len(joined_df)

(790, 790)

In [21]:
joined_df.to_csv(DATA_PATH / "final_runs_table_fixed.csv", index=False)